# Character-Level Transformer for Hindi Transliteration

Encoder-decoder Transformer trained on the Dakshina Hindi transliteration corpus.
Input: Latin/Roman transliteration. Output: Devanagari Hindi.

Set the `DAKSHINA_DATA_DIR` environment variable (or edit `DATA_DIR` below) before running.

In [1]:
# Install dependencies if needed (uncomment on Colab/Kaggle)
# !pip install wandb nltk tqdm

import os
import math
import random
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Data Loading and Preprocessing

In [2]:
from google.colab import files
import io

print('Upload 3 TSV files: train, dev, and test')

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
uploaded = files.upload()

def load_tsv_pairs(content):
    pairs = []
    for line in io.StringIO(content.decode('utf-8')):
        parts = line.strip().split('\t')
        if len(parts) < 3:
            continue
        native = parts[0].strip()
        roman = parts[1].strip()
        if roman and native:
            pairs.append((roman, native))
    if not pairs:
        raise ValueError('No data loaded from file')
    return pairs

train_pairs = None
dev_pairs = None
test_pairs = None

for filename, content in uploaded.items():
    if 'train' in filename.lower():
        train_pairs = load_tsv_pairs(content)
    elif 'dev' in filename.lower() or 'valid' in filename.lower():
        dev_pairs = load_tsv_pairs(content)
    elif 'test' in filename.lower():
        test_pairs = load_tsv_pairs(content)
    else:
        if train_pairs is None:
            train_pairs = load_tsv_pairs(content)
        elif dev_pairs is None:
            dev_pairs = load_tsv_pairs(content)
        elif test_pairs is None:
            test_pairs = load_tsv_pairs(content)

assert train_pairs is not None and dev_pairs is not None and test_pairs is not None,     'Could not identify all three splits. Make sure filenames contain train/dev/test.'

print(f'Train: {len(train_pairs)}  Dev: {len(dev_pairs)}  Test: {len(test_pairs)}')
print('Sample:', train_pairs[:3])


Upload 3 TSV files: train, dev, and test


Saving hi.translit.sampled.dev.tsv to hi.translit.sampled.dev.tsv
Saving hi.translit.sampled.test.tsv to hi.translit.sampled.test.tsv
Saving hi.translit.sampled.train.tsv to hi.translit.sampled.train.tsv
Train: 51200  Dev: 4096  Test: 4096
Sample: [('shastragaar', 'शस्त्रागार'), ('bindhya', 'बिन्द्या'), ('kirankant', 'किरणकांत')]


In [3]:
def build_vocab(pairs):
    src_chars = set()
    tgt_chars = set()
    for src, tgt in pairs:
        src_chars.update(src)
        tgt_chars.update(tgt)
    tokens = SPECIAL_TOKENS + sorted(src_chars)
    src2idx = {c: i for i, c in enumerate(tokens)}
    tokens = SPECIAL_TOKENS + sorted(tgt_chars)
    tgt2idx = {c: i for i, c in enumerate(tokens)}
    idx2src = {i: c for c, i in src2idx.items()}
    idx2tgt = {i: c for c, i in tgt2idx.items()}
    return src2idx, idx2src, tgt2idx, idx2tgt


src2idx, idx2src, tgt2idx, idx2tgt = build_vocab(train_pairs + dev_pairs)
PAD, SOS, EOS, UNK = (
    tgt2idx["<PAD>"], tgt2idx["<SOS>"],
    tgt2idx["<EOS>"], tgt2idx["<UNK>"]
)
SRC_VOCAB_SIZE = len(src2idx)
TGT_VOCAB_SIZE = len(tgt2idx)
MAX_LEN = 32

print(f"Source vocab: {SRC_VOCAB_SIZE}  Target vocab: {TGT_VOCAB_SIZE}  Max len: {MAX_LEN}")


Source vocab: 30  Target vocab: 68  Max len: 32


In [4]:
class TransliterationDataset(Dataset):
    def __init__(self, pairs, max_len=MAX_LEN):
        self.pairs = pairs
        self.max_len = max_len

    def encode(self, text, vocab, add_special=True):
        ids = [vocab.get(c, UNK) for c in text[: self.max_len - 2]]
        if add_special:
            ids = [SOS] + ids + [EOS]
        else:
            ids = ids + [EOS]
        return torch.LongTensor(ids)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        return (
            self.encode(src, src2idx, add_special=False),
            self.encode(tgt, tgt2idx, add_special=True),
        )


def collate_fn(batch):
    srcs, tgts = zip(*batch)
    src_max = max(len(s) for s in srcs)
    tgt_max = max(len(t) for t in tgts)
    src_pad = torch.full((len(srcs), src_max), PAD, dtype=torch.long)
    tgt_pad = torch.full((len(tgts), tgt_max), PAD, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, : len(s)] = s
        tgt_pad[i, : len(t)] = t
    return src_pad, tgt_pad


BATCH_SIZE = 128
train_loader = DataLoader(TransliterationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(TransliterationDataset(dev_pairs), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(TransliterationDataset(test_pairs), batch_size=BATCH_SIZE,
                         shuffle=False, collate_fn=collate_fn)
print(f"DataLoaders ready: {len(train_loader)} train batches")


DataLoaders ready: 400 train batches


## Transformer Model

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TransformerSeq2Seq(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=128,
                 nhead=4, num_layers=3, dim_ff=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD)
        self.pos_encoding = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        src_key_padding_mask = (src == PAD)
        tgt_key_padding_mask = (tgt == PAD)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)

        src_emb = self.pos_encoding(self.src_embed(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoding(self.tgt_embed(tgt) * math.sqrt(self.d_model))

        output = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.fc_out(output)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerSeq2Seq(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE).to(DEVICE)
optimizer = Adam(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {DEVICE} | Parameters: {total_params:,}")


Device: cuda | Parameters: 1,015,620


## Training Loop

In [6]:
EPOCHS = 20

def train_epoch(loader):
    model.train()
    total_loss = 0.0
    for src, tgt in tqdm(loader, desc="Training", leave=False):
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        optimizer.zero_grad()
        output = model(src, tgt[:, :-1])
        loss = criterion(
            output.reshape(-1, TGT_VOCAB_SIZE),
            tgt[:, 1:].reshape(-1)
        )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(loader):
    model.eval()
    total_loss = 0.0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        output = model(src, tgt[:, :-1])
        loss = criterion(
            output.reshape(-1, TGT_VOCAB_SIZE),
            tgt[:, 1:].reshape(-1)
        )
        total_loss += loss.item()
    return total_loss / len(loader)


best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(train_loader)
    val_loss = evaluate_loss(dev_loader)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "transformer_best.pt")
        print("  Saved best model.")


Training:   0%|          | 0/400 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01/20 | Train Loss: 2.2680 | Val Loss: 1.5421
  Saved best model.


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 02/20 | Train Loss: 1.6107 | Val Loss: 1.0848
  Saved best model.


...

Epoch 18/20 | Train Loss: 0.4651 | Val Loss: 0.4015
  Saved best model.


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 19/20 | Train Loss: 0.4525 | Val Loss: 0.3939
  Saved best model.


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Epoch 20/20 | Train Loss: 0.4406 | Val Loss: 0.3928
  Saved best model.


## Greedy Inference and Evaluation

In [7]:
@torch.no_grad()
def translate_greedy(src_text, max_len=MAX_LEN):
    model.eval()
    src_ids = [src2idx.get(c, UNK) for c in src_text]
    src_tensor = torch.LongTensor(src_ids + [EOS]).unsqueeze(0).to(DEVICE)
    ys = torch.LongTensor([[SOS]]).to(DEVICE)

    for _ in range(max_len - 1):
        output = model(src_tensor, ys)
        next_token = output[:, -1].argmax(-1).unsqueeze(0)
        ys = torch.cat([ys, next_token], dim=1)
        if next_token.item() == EOS:
            break

    ids = ys[0, 1:].tolist()
    if ids and ids[-1] == EOS:
        ids = ids[:-1]  # drop EOS; keep all chars if max_len was reached
    return "".join(idx2tgt[i] for i in ids)


@torch.no_grad()
def evaluate_test(loader):
    model.eval()
    correct_words = total_words = correct_chars = total_chars = 0
    for src_batch, tgt_batch in tqdm(loader, desc="Testing"):
        src_batch = src_batch.to(DEVICE)
        for i in range(src_batch.size(0)):
            src_ids = src_batch[i][src_batch[i] != PAD].tolist()
            src_text = "".join(idx2src.get(j, "") for j in src_ids[:-1])
            pred = translate_greedy(src_text)

            true_ids = [
                j.item() for j in tgt_batch[i]
                if j.item() not in (PAD, SOS, EOS)
            ]
            truth = "".join(idx2tgt[j] for j in true_ids)

            if pred == truth:
                correct_words += 1
            total_words += 1
            min_len = min(len(pred), len(truth))
            correct_chars += sum(a == b for a, b in zip(pred, truth[:min_len]))
            total_chars += len(truth)

    word_acc = correct_words / total_words if total_words else 0
    char_acc = correct_chars / total_chars if total_chars else 0
    print(f"Word Accuracy: {word_acc:.4f} | Char Accuracy: {char_acc:.4f}")
    return {"word_accuracy": word_acc, "char_accuracy": char_acc}


# Load best model and evaluate
model.load_state_dict(torch.load("transformer_best.pt", map_location=DEVICE))
metrics = evaluate_test(test_loader)

# Show a few predictions
for roman, deva in test_pairs[:10]:
    print(f"{roman:>15s} -> {translate_greedy(roman):<15s} (truth: {deva})")

Testing:   0%|          | 0/32 [00:00<?, ?it/s]

Word Accuracy: 0.3433 | Char Accuracy: 0.6737
        thermax -> थेर्मैक्स       (truth: थरमैक्स)
      sikhaaega -> सिखाएगा         (truth: सिखाएगा)
          learn -> लीर्न           (truth: लर्न)
       twitters -> ट्विटर्स        (truth: ट्विटर्स)
    tirunelveli -> तिरुनेलवेली     (truth: तिरुनेलवेली)
   independence -> इंडेपेंडेंस     (truth: इंडिपेंडेंस)
      speshiyon -> स्पेशियों       (truth: स्पेशियों)
        shurooh -> शुरूह           (truth: शुरूः)
       kolhapur -> कोल्हापुर       (truth: कोल्हापुर)
          ajhar -> अझर             (truth: अजहर)
